# Tutorial Classification

> Tutorial Classification

In [ ]:
#| default_exp tutorial_1

In [ ]:
from bioMONAI.data import *
from bioMONAI.transforms import *
from bioMONAI.core import *
from bioMONAI.core import Path
from bioMONAI.data import get_image_files, get_target, RandomSplitter
from bioMONAI.nets import BasicUNet, DynUNet
from bioMONAI.losses import *
from bioMONAI.metrics import *
from bioMONAI.datasets import download_medmnist

from monai.utils import set_determinism
from monai.transforms import ScaleIntensity

set_determinism(0)

In [ ]:
device = get_device()
print(device)

### Download Data

In [ ]:
image_path = '../_data/medmnist_data/'
info = download_medmnist('bloodmnist', image_path, download_only=True)
info

### Create Dataloader

In [ ]:
from fastai.vision.all import *

In [ ]:
bs, size = 4, 128

path = Path(image_path)
path_train = path/'train'
path_val = path/'val'


data_ops = {
    'blocks':       (BioImageBlock(cls=BioImageStack), CategoryBlock(vocab=info['label'])),
    'get_items':    get_image_files,
    'get_y':        parent_label,
    'splitter':     GrandparentSplitter(),
    'item_tfms':    [],
    'bs': bs,
}

data = get_dataloader(
    path, 
    show_summary=True,
    **data_ops,
    )

# print length of training and validation datasets
print('train images:', len(data.train_ds.items), '\nvalidation images:', len(data.valid_ds.items))

In [ ]:
data.show_batch(max_n=4)

### Load and train a 2D model

In [ ]:
model = BasicUNet
 
loss = MSSSIML1Loss(2, levels=2) 
metrics = [SSIMMetric, MSELoss]

trainer = fastTrainer(data, model, loss_fn=loss, metrics=metrics, show_summary=False)

In [ ]:
trainer.fit_flat_cos(1)

In [ ]:
trainer.show_results(cmap='gray')

In [ ]:
# trainer.save('tmp-model')

### Test data 
Evaluate the performance of the selected model on unseen data.
It’s important to not touch this data until you have fine tuned your model to get an unbiased evaluation!